In [ ]:
import pathlib
import jupedsim as jps
import pedpy
from numpy.random import normal  # normal distribution of free movement speed
from shapely import Polygon, GeometryCollection, intersection
from matplotlib.patches import Circle
import random

# reference from: (https://www.jupedsim.org/stable/notebooks/queues_waiting.html)

number_of_queues = 10 # must be even

# dimensions of shape
wings_length = 14
base_length = 20

# length of square side blocks
obs_block_length = 3

# crowd controlling variables
spawn_interval = 200 # spawning of agents
num_agents = 15 # number of agents max per section
distance_of_agents = 1.0 # distance of agents to each other, lower = tighter crowd
# burst_size = 5
# ^^ still tentative, idea is to have bursts of agents spawn from specific locations using for loops for every n intervals

# walkable areas
north_landing = Polygon([(0, wings_length+base_length), (20, wings_length+base_length), (20, wings_length*2+base_length), (0, wings_length*2+base_length)])
main_landing = Polygon([(-12, wings_length), (32, wings_length), (32, wings_length+base_length), (-12, wings_length+base_length)])
south_landing = Polygon([(0, 0), (20, 0), (20, wings_length), (0, wings_length)])

# obstacles
center_booth = Polygon([(6, 21), (14, 21), (14, 29), (6, 29)])

west_block_1 = Polygon([(-3, wings_length), (-3, wings_length + obs_block_length), (-3 - obs_block_length, wings_length + obs_block_length), (-3 - obs_block_length, wings_length)])
west_block_2 = Polygon([(-3, wings_length + base_length), (-3, wings_length + base_length - obs_block_length), (-3 - obs_block_length, wings_length +base_length - obs_block_length), (-3 - obs_block_length, wings_length + base_length)])

east_block_1 = Polygon([(23, wings_length), (23, wings_length + obs_block_length), (23 + obs_block_length, wings_length + obs_block_length), (23 + obs_block_length, wings_length)])
east_block_2 = Polygon([(23, wings_length + base_length), (23, wings_length + base_length - obs_block_length), (23 + obs_block_length, wings_length +base_length - obs_block_length), (23 + obs_block_length, wings_length + base_length)])

# switches
switch_n = (10, 34)
switch_s = (10, 14)

# helper function for makking queues from left to right
def make_horizontal_queues(number_of_queues : int,
                           line_size : int,
                           length: int,
                           start_pos_x : int,
                           start_pos_y : int) -> list[list[tuple[int, int]]]:
    waiting_queues = []
    increments = length / (number_of_queues / 2) # for each queue n, offset by increments
    middle = int(number_of_queues / 2) # middle of the number of queues
    start_y_dec = start_pos_y # temp var to store start_pos_y for 2 for loops
    for _ in range(middle):
        waiting_list = []
        for j in range(line_size):
            waiting_list.append((start_pos_x + j, start_y_dec)) # append it + i'th idx
        start_y_dec -= increments
        waiting_queues.append(waiting_list)
        
    start_y_dec = start_pos_y # reset
    for _ in range(middle):
        waiting_list = []
        for j in range(line_size):
            waiting_list.append((start_pos_x - j + 36, start_y_dec)) # 36 is hard coded for now, change if length also changes (chopped)
        start_y_dec -= increments
        waiting_queues.append(waiting_list)
        
    return waiting_queues

waiting_positions_queues = make_horizontal_queues(number_of_queues, 3, 15, -8, 30)

area = GeometryCollection(north_landing.union(main_landing).union(south_landing).difference(center_booth).difference(west_block_1).difference(west_block_2).difference(east_block_1).difference(east_block_2))
walkable_area = pedpy.WalkableArea(area)
pedpy.plot_walkable_area(walkable_area=walkable_area).set_aspect("equal")

north_spawn_pos = []
south_spawn_pos = []

def spawn_in_area(area : Polygon, target_list : list):
    target_list.extend(
        jps.distributions.distribute_by_number(
            polygon=area,
            number_of_agents=num_agents,
            distance_to_agents=distance_of_agents,
            distance_to_polygon=0.2,
            seed=None,
        )
    )

# entrances
south_left_entrance = Polygon([(0, 0), (3, 0), (3, 10), (0, 10)])
south_right_entrance = Polygon([(17, 0), (20, 0), (20, 10), (17, 10)])
north_left_entrance = Polygon([(0, 38), (3, 38), (3, 48), (0, 48)])
north_right_entrance = Polygon([(17, 38), (20, 38), (20, 48), (17, 48)])

spawn_in_area(south_left_entrance, south_spawn_pos)
spawn_in_area(south_right_entrance, south_spawn_pos)
spawn_in_area(north_left_entrance, north_spawn_pos)
spawn_in_area(north_right_entrance, north_spawn_pos)

# exits
right_exit = Polygon([(30, 14), (32, 14), (32, 34), (30, 34)])
left_exit = Polygon([(-12, 14), (-10, 14), (-10, 34), (-12, 34)])

queues_per_side = int(number_of_queues / 2)
exit_areas = [left_exit] * queues_per_side + [right_exit] * queues_per_side # Reordered to match queue order

def plot_simulation_configuration_with_queues(
    walkable_area, spawning_area, starting_positions, exit_areas, waiting_positions_queues
):
    axes = pedpy.plot_walkable_area(walkable_area=walkable_area)
    axes.fill(*spawning_area.exterior.xy, color="lightgrey")
    for exit_area in exit_areas:
        axes.fill(*exit_area.exterior.xy, color="indianred")
    axes.scatter(*zip(*starting_positions))
    
    # Plot queue positions
    for queue_positions in waiting_positions_queues:
        axes.scatter(*zip(*queue_positions), s=10, color='blue')
    
    axes.set_xlabel("x/m")
    axes.set_ylabel("y/m")
    axes.set_aspect("equal")
    return axes

plot_simulation_configuration_with_queues(
    walkable_area, south_right_entrance, north_spawn_pos, exit_areas, waiting_positions_queues
)

trajectory_file = "corner_with_queues.sqlite"  # output file
simulation = jps.Simulation(
    model=jps.CollisionFreeSpeedModel(),  # Changed to match queue example
    geometry=area,
    trajectory_writer=jps.SqliteTrajectoryWriter(
        output_file=pathlib.Path(trajectory_file)
    ),
)

# Create queue stages
queue_stages_ids = [simulation.add_queue_stage(queue_pos) for queue_pos in waiting_positions_queues]
queue_stages = [simulation.get_stage(stage_id) for stage_id in queue_stages_ids]

# Create exit stages
exit_ids = []
for exit_area in exit_areas:
    exit_ids.append(simulation.add_exit_stage(exit_area.exterior.coords[:-1]))

# Create distribution waypoints (switches)
south_switch_id = simulation.add_waypoint_stage(switch_s, 1)
north_switch_id = simulation.add_waypoint_stage(switch_n, 1)

def create_journey_with_queues(simulation, switch_id, queue_stages_ids, exit_ids):
    """Create a journey that goes through distribution point to queues to exits."""
    journey = jps.JourneyDescription(queue_stages_ids + exit_ids + [switch_id]) # all possible nodes in the simulation
    
    # From switch to least targeted queue
    journey.set_transition_for_stage(
        switch_id,
        jps.Transition.create_least_targeted_transition(queue_stages_ids),
    )
    
    # From each queue to corresponding exit
    for queue_id, exit_id in zip(queue_stages_ids, exit_ids):
        journey.set_transition_for_stage(
            queue_id,
            jps.Transition.create_fixed_transition(exit_id)
        )
    
    journey_id = simulation.add_journey(journey)
    return journey_id

# Create journeys
south_journey_id = create_journey_with_queues(simulation, south_switch_id, queue_stages_ids, exit_ids)
north_journey_id = create_journey_with_queues(simulation, north_switch_id, queue_stages_ids, exit_ids)

south_index = 0
north_index = 0

south_speeds = normal(1.34, 0.05, len(south_spawn_pos))
north_speeds = normal(1.34, 0.05, len(north_spawn_pos))

max_frames = 10000
frame = 0

# Queue management variables
queue_started = [False for i in range(number_of_queues)]
queue_offsets = [0 for i in range(number_of_queues)]

while frame < max_frames:
    
    # Queue management (similar to original queue example)
    for i in range(number_of_queues):
        if queue_stages[i].count_enqueued() == 0:
            queue_started[i] = False
        elif not queue_started[i] and queue_stages[i].count_enqueued() > 0:
            queue_started[i] = True
            queue_offsets[i] = frame
        elif (
            queue_started[i]
            and (frame - queue_offsets[i]) % 1000 == 0  # Process every 500 frames
        ):
            queue_stages[i].pop(1)  # Remove one agent from queue

    if frame % spawn_interval == 0:

        # SOUTH SPAWN
        if south_index < len(south_spawn_pos):
            pos = south_spawn_pos[south_index]
            v0 = south_speeds[south_index]

            simulation.add_agent(
                jps.CollisionFreeSpeedModelAgentParameters(  # Changed to match queue model
                    journey_id=south_journey_id,
                    stage_id=south_switch_id,
                    position=pos,
                    desired_speed=v0,
                )
            )

            south_index += 1

        # NORTH SPAWN
        if north_index < len(north_spawn_pos):
            pos = north_spawn_pos[north_index]
            v0 = north_speeds[north_index]

            simulation.add_agent(
                jps.CollisionFreeSpeedModelAgentParameters(  # Changed to match queue model
                    journey_id=north_journey_id,
                    stage_id=north_switch_id,
                    position=pos,
                    desired_speed=v0,
                )
            )

            north_index += 1

    simulation.iterate()
    frame += 1

from jupedsim.internal.notebook_utils import animate, read_sqlite_file

trajectory_data, walkable_area = read_sqlite_file(trajectory_file)
animate(trajectory_data, walkable_area, every_nth_frame=10)
